# Amazon Bedrock: Foundation Models, Model Lifecycle, and Inference Parameters

## 1. Theory & Real-World Context

**Foundation Models (FMs)** are large-scale machine learning models trained on vast, unlabelled datasets. Unlike traditional ML models built for narrow tasks (e.g., predicting house prices), FMs learn general data representations, allowing them to be fine-tuned or prompted for a wide array of downstream tasks (text generation, coding, image creation).

Amazon Bedrock provides a unified, serverless API to access industry-leading FMs (from Anthropic, Meta, Mistral, Cohere, and Amazon) along with an **Amazon Bedrock Marketplace** for specialized, domain-specific models (e.g., biomedical or financial FMs).

### The Bedrock Model Lifecycle

Understanding the lifecycle is critical for production AI pipelines:

1. **Active:** The recommended, fully supported version.
2. **Legacy:** An older version being phased out. Existing apps continue to work (for at least 6 months), but no new Provisioned Throughput can be allocated. A **Public Extended Access Period** may apply, introducing higher inference costs to incentivize migration.
3. **End-of-Life (EOL):** The model is fully retired. Requests will fail. **Migration is manual**—you must update your code to point to a new model ID.

### Inference Parameters (The "Missing Tabs")

Inference parameters are the levers you pull to shape the output of a foundation model.

**Randomness & Diversity:**

* **Temperature (0.0 to 1.0+):** Modulates the probability distribution of the next token. Low values (~0.0) make the model highly deterministic and factual. High values (~0.9) flatten the probabilities, allowing the model to pick less common words, generating more creative or varied text.
* **Top K:** An absolute cutoff. The model restricts its selection pool to only the *K* most probable next tokens, discarding the long tail of unlikely words.
* **Top P (Nucleus Sampling):** A dynamic cutoff. The model restricts its selection pool to the smallest set of tokens whose cumulative probabilities equal *P*. For example, `Top P = 0.9` means the model considers only the tokens making up the top 90% of the probability mass.

**Length & Boundaries:**

* **Response Length (Max Tokens):** A hard limit on the maximum number of tokens the model can generate in a single response.
* **Length Penalty:** A modifier that discourages the model from generating excessively long outputs without hitting a hard, abrupt cutoff.
* **Stop Sequences:** Specific string patterns (e.g., `\n\nUser:`, `###`) that force the model to immediately halt generation when encountered, preventing run-on text or hallucinated conversations.

> **Real-World AI Engineering Context:** For strict data extraction workflows (e.g., pulling JSON entities from PDFs), you set `Temperature = 0.0` and `Top P = 0.0` to force maximum determinism. For generating synthetic training data or creative copywriting, you increase `Temperature = 0.7 - 0.9` to introduce variance.

---

## 2. Visual Architecture

```mermaid
flowchart TD
    A[Model Calculates Probabilities<br/>for all possible next tokens] --> B{Top K Filter}
    B -- Keeps the Top 'K' tokens by rank --> C{Top P Filter}
    C -- Keeps tokens until cumulative<br/>probability equals 'P' --> D{Temperature Scaling}
    D -- Reshapes probability curve<br/>(Sharpens or Flattens) --> E[Sample Next Token]
    
    E --> F{Stop Sequence<br/>Generated?}
    F -- Yes --> G([End Generation])
    F -- No --> H{Max Token Limit<br/>Reached?}
    
    H -- Yes --> G
    H -- No --> A

```

---

## 3. Console Hands-On

**Objective:** Observe the impact of inference parameters directly in the console.

1. Navigate to the **Amazon Bedrock Console** $\rightarrow$ **Playgrounds** $\rightarrow$ **Chat**.
2. Click **Select model** and choose **Anthropic $\rightarrow$ Claude 3 Haiku**.
3. In the right-hand **Configurations** panel, set **Temperature** to `0.0`, **Top P** to `0.001`, and **Maximum length** to `500`.
4. Enter the prompt: `Write a one-paragraph story about an AI discovering the internet.` Note the straightforward, dry response.
5. Change **Temperature** to `1.0` and **Top P** to `0.999`. Rerun the prompt. Notice the drastic shift in vocabulary, tone, and creativity.
6. Under **Stop sequences**, type the word `data` and press Enter. Rerun the prompt. The model will abruptly stop generating the exact moment it tries to output the word "data".

---

## 5. Practice Challenges

### Challenge 1: The Hallucination Breaker

**Scenario:** You are building an AI data pipeline that converts raw support emails into strict SQL queries using Amazon Bedrock. Currently, the model is generating valid SQL, but occasionally appends conversational fluff like *"Here is your query:"* or continues generating a second, unrequested query.

* **Task:** Identify the exact three inference parameters you should adjust (and their values) to force strict, code-only determinism and halt generation after the first query finishes.

### Challenge 2: Programmatic Lifecycle Monitoring

**Scenario:** Your company depends heavily on `anthropic.claude-v2` for a production pipeline.

* **Task:** Using the `boto3.client('bedrock')` control plane client, write a function that calls `list_foundation_models()`. Filter the response to find your specific model ID and programmatically print an alert if the `modelLifecycle.status` transitions from `ACTIVE` to `LEGACY`.

---

## 6. Industry Standards & Tips

* **The "One Lever" Rule:** Never tune Temperature, Top P, and Top K simultaneously. Industry standard is to leave Top K and Top P at their defaults and only adjust **Temperature**. If you require even finer control, adjust Top P, but leave Top K alone.
* **Lifecycle CI/CD Checks:** Never let an EOL date catch you off guard. Write an automated integration test that queries the `GetFoundationModel` API for your active models. If a model transitions to `Legacy`, fail the build gracefully with a warning to the MLOps team that a 6-month migration clock has started.
* **Prompt vs. Parameters:** If your model is consistently outputting the wrong format, inference parameters cannot fix it. Parameters control *how* the model selects words, not *what* the model understands. Fix formatting issues with Prompt Engineering (e.g., Few-Shot examples); use inference parameters strictly for controlling variance.

---

## 7. Interview Q&A (Targeted for AI Engineers)

### Q1: How do Top K and Top P differ mathematically when shaping a model's response?

> **Answer:** Top K is a static filter that restricts the model's vocabulary to a fixed number of tokens (e.g., the top 50), regardless of how much probability mass they actually hold. Top P (Nucleus Sampling) is dynamic; it restricts the vocabulary to the smallest set of tokens whose cumulative probability reaches the threshold $P$. If the model is highly confident in one word (e.g., 95% probability), a Top P of 0.90 will only consider that single word, ignoring the rest of the pool.

### Q2: We are heavily relying on an Amazon Bedrock foundation model that just transitioned to the `Legacy` state. What are the immediate impacts on our infrastructure, and what is our timeline?

> **Answer:** Existing API calls will not break immediately; the model remains available for at least 6 months. However, we cannot purchase any new Provisioned Throughput for it, and no new AWS accounts can adopt it. If it enters the Public Extended Access Period, we may incur higher inference costs. We must prioritize migrating our code to a newer, `Active` model before the End-of-Life (EOL) date, after which all API requests will return hard failures.

### Q3: You want to prevent a model from generating overly long, verbose responses. Should you rely entirely on the `Max Tokens` parameter?

> **Answer:** No. While `Max Tokens` provides a hard safety limit to prevent runaway API costs, relying on it alone will result in the model abruptly stopping mid-sentence when the limit is hit. Instead, you should combine a reasonable `Max Tokens` limit with a higher **Length Penalty** (to naturally encourage the model to output shorter sequences) and configure proper **Stop Sequences** so the model knows exactly when its generation task is complete.

---

## Boto3 Implementation (optional)

Below is a production-ready script using the modern Bedrock `Converse` API, which standardizes inference parameters (Temperature, Top P, Max Tokens, Stop Sequences) across all foundation models.

```python
import logging
import boto3
from botocore.exceptions import ClientError

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

class BedrockInferenceManager:
    """Manages AI model invocations using standardized Bedrock inference parameters."""
    
    def __init__(self, region_name: str = "us-east-1"):
        self.bedrock_runtime = boto3.client("bedrock-runtime", region_name=region_name)

    def generate_deterministic_response(self, prompt: str, model_id: str) -> str:
        """
        Invokes a Bedrock model configured for deterministic, factual output 
        (e.g., for RAG pipelines or data extraction).
        """
        # The Converse API standardizes parameters across all model providers
        inference_config = {
            "temperature": 0.0,      # Maximum strictness; no creativity
            "topP": 0.1,             # Restrict to the highest probability tokens
            "maxTokens": 512,        # Cap the output length
            "stopSequences": ["\n\nUser:", "```\n\n"] # Stop if it hallucinates boundaries
        }
        
        messages = [{"role": "user", "content": [{"text": prompt}]}]

        try:
            logger.info(f"Invoking {model_id} with deterministic settings...")
            response = self.bedrock_runtime.converse(
                modelId=model_id,
                messages=messages,
                inferenceConfig=inference_config
            )
            
            output_text = response['output']['message']['content'][0]['text']
            stop_reason = response['stopReason']
            
            logger.info(f"Generation stopped due to: {stop_reason}")
            return output_text
            
        except ClientError as e:
            logger.error(f"Bedrock invocation failed: {e.response['Error']['Message']}")
            raise

if __name__ == "__main__":
    MODEL_ID = "anthropic.claude-3-haiku-20240307-v1:0"
    manager = BedrockInferenceManager()
    
    result = manager.generate_deterministic_response(
        prompt="Extract the key entities from the following text and format as JSON: [Text body]",
        model_id=MODEL_ID
    )
    print(result)

```

# Amazon Bedrock Foundation Models: I/O Configurations & Parameter Tuning

## 1. Theory & Real-World Context

Amazon Bedrock hosts a diverse catalog of Foundation Models (FMs), each optimized for specific generative AI modalities. While standard parameters (`temperature`, `topP`, `maxTokens`) apply universally, mastering model-specific configurations and API patterns is critical for production AI pipelines.

### Model Families & AI Engineering Applications

* **Amazon Titan:** Native AWS models. Includes **Titan Text** (general NLP), **Titan Embeddings** (text and multimodal vectors for Retrieval-Augmented Generation / RAG), and **Titan Image Generator v1/v2** (inpainting, outpainting, background removal).
* **Amazon Nova:** AWS's newest family optimized for speed and cost.
* *Understanding:* **Micro** (text-only, lowest latency), **Lite** (low-cost multimodal), **Pro** (advanced multimodal reasoning), **Premier** (highest accuracy, supports custom model distillation).
* *Creative:* **Canvas** (image generation with watermarking), **Reel** (video generation), **Sonic** (low-latency, real-time speech-to-speech).


* **AI21 Labs (Jurassic-2 & Jamba):** Enterprise-focused models. **Jamba-Instruct** offers a massive 256K context window. **Jurassic-2** introduces advanced repetition management via `presencePenalty`, `countPenalty`, and `frequencyPenalty`.
* **Anthropic Claude 4 Family:** Industry-leading multimodal reasoning models (Opus 4 / 4.1, Sonnet 4 / 4.5, Haiku 4.5). Introduces **Extended Thinking** (`thinking`), allowing the model to consume a dedicated token budget for step-by-step internal reasoning before returning an answer.
* **Stability AI (SDXL):** Text-to-image models focusing on prompt adherence (`cfg_scale`), sampling iterations (`steps`), and reproducibility (`seed`).

### API Paradigms

1. **Converse API:** The modern, unified Bedrock interface. It standardizes the `messages` array, `system` prompts, and inference parameters (`temperature`, `topP`, `maxTokens`) across all supported text/multimodal models (Claude, Nova Understanding, Jamba).
2. **InvokeModel API:** The legacy/bespoke interface. Requires passing a model-specific, raw JSON payload in the `body`. Mandatory for embeddings (Titan) and creative/image generation models (SDXL, Titan Image, Nova Canvas) where the normalized `messages` format does not apply.

---

## 2. Visual Architecture

```mermaid
flowchart TD
    subgraph Client Application
        App[AI Engineering Application]
    end

    subgraph Bedrock API Gateway
        CAPI[Converse API<br/>Unified Standard]
        IMAPI[InvokeModel API<br/>Bespoke JSON Body]
    end

    subgraph Text & Multimodal Understanding
        C4[Anthropic Claude 4 Family<br/>Extended Thinking]
        NVU[Amazon Nova<br/>Micro, Lite, Pro, Premier]
        JMB[AI21 Jamba-Instruct<br/>256K Context]
    end

    subgraph Embeddings & Creative Generation
        TIT[Titan Embeddings v2<br/>Titan Multimodal G1]
        SDX[Stability AI SDXL<br/>cfg_scale, steps, seed]
        NVC[Amazon Nova Canvas/Reel/Sonic]
    end

    App -- "Standardized Messages" --> CAPI
    App -- "Raw JSON Payloads" --> IMAPI

    CAPI --> C4
    CAPI --> NVU
    CAPI --> JMB

    IMAPI --> TIT
    IMAPI --> SDX
    IMAPI --> NVC

```

---

## 3. Console Hands-On

**Objective:** Test Claude 4's Extended Thinking and SDXL's `cfg_scale` in the Bedrock Playground.

1. Navigate to the **Amazon Bedrock Console** $\rightarrow$ **Playgrounds** $\rightarrow$ **Chat**.
2. Click **Select model** $\rightarrow$ **Anthropic** $\rightarrow$ **Claude 4.5 Sonnet** (or Opus 4).
3. In the right-hand **Configurations** panel, locate **Extended Thinking** and enable it. Set the thinking token budget to `1024`.
4. Submit a complex logic puzzle prompt. Notice how the model generates an internal "thinking" block before yielding the final text.
5. Switch the playground type to **Image** (left navigation pane). Select **Stability AI** $\rightarrow$ **SDXL**.
6. Prompt: `A futuristic cyberpunk city skyline`. Under Configurations, change the **Prompt strength (cfg_scale)** to `10.0` (high adherence to prompt) vs `2.0` (high randomness). Compare the two resulting images.

---

## 5. Practice Challenges

### Challenge 1: The Unified RAG Pipeline

**Scenario:** You are building a Retrieval-Augmented Generation (RAG) system. You need to convert a user's search query into an embedding, and then pass that context into a language model.

* **Task:** Write a Python function that first uses `invoke_model` to hit `amazon.titan-embed-text-v2:0` (generating a 1024-dimension vector with `normalize: True`). Then, pass a summary of the retrieved data to `amazon.nova-lite-v1:0` using the `converse` API.

### Challenge 2: AI21 Repetition Penalization

**Scenario:** An automated marketing copy generator using AI21 Jurassic-2 Ultra frequently repeats specific promotional adjectives across long product descriptions.

* **Task:** Construct an `invoke_model` JSON payload for the Jurassic-2 model that drastically reduces the likelihood of repeating words by heavily weighting the `frequencyPenalty` while exempting emojis from the penalty using the `applyToEmojis` parameter.

---

## 6. Industry Standards & Tips

* **API Selection Standard:** Default to the **Converse API** for all text-in/text-out and image-in/text-out multimodal tasks. It future-proofs your code, allowing you to swap from Claude 4 to Nova Premier to Jamba-Instruct by merely changing the `modelId` string, without rewriting your payload structures.
* **Embedding Normalization:** When using Titan Embeddings v2, setting `"normalize": True` standardizes vector lengths to 1. This optimizes distance calculations (like Cosine Similarity) in downstream vector databases like Amazon OpenSearch Serverless, significantly speeding up semantic searches.
* **Extended Thinking Cost Trap:** Claude 4's `thinking` parameter consumes tokens from your generated output budget. Because Bedrock bills per input and output token, excessive thinking budgets on large volume tasks will exponentially increase your inference costs. Use it strictly for complex multi-step reasoning, not standard Q&A.
* **Deterministic Imagery (SDXL/Titan):** In production AI engineering, always pass a discrete integer to the `seed` parameter and log it in your database. If a user generates a highly successful image and wants to slightly tweak it via a prompt modification, reusing the exact same seed ensures the base composition remains identical.

---

## 7. Interview Q&A (Targeted for AI Engineers)

### Q1: Why would an AI Engineering team choose the Bedrock `Converse` API over the `InvokeModel` API?

> **Answer:** The `Converse` API abstracts away the bespoke JSON payload schemas required by individual model providers. It normalizes message formatting, system prompts, and common inference parameters (like Temperature and Max Tokens). This allows engineers to hot-swap models (e.g., Anthropic to Amazon Nova) by just changing a single `modelId` variable, radically simplifying application code and MLOps testing.

### Q2: Explain the difference between AI21's `presencePenalty` and `frequencyPenalty`.

> **Answer:** `presencePenalty` is a boolean-style flat penalty; it lowers the probability of a token being generated if it has appeared *at least once* previously, regardless of how many times it appeared. It encourages broad topic diversity. `frequencyPenalty` is proportional; the penalty increases exponentially the more frequently a specific token appears. It prevents the model from getting stuck in repetitive loops.

### Q3: How does Anthropic Claude 4's `thinking` parameter fundamentally change token generation?

> **Answer:** Instead of immediately predicting the next word of the final response, enabling `thinking` allocates a dedicated token budget for the model to perform internal, step-by-step reasoning. These "thinking tokens" are processed hidden from the final user output. It drastically improves accuracy on complex math, logic, and coding tasks at the cost of higher latency and increased token expenditure.

### Q4: When evaluating the Amazon Nova family for a workload that parses document images and charts at high volume, which model should you choose and why?

> **Answer:** **Amazon Nova Lite**. Nova Micro is text-only, so it cannot process document images. While Nova Pro or Premier can process images, they are over-provisioned for standard chart parsing. Nova Lite balances multimodal visual comprehension capabilities with the affordability and high-throughput required for large-scale document analysis.

## 4. Boto3 Implementation

Production-grade script demonstrating both the unified `Converse` API for Claude 4 and the raw `InvokeModel` API for Stability AI image generation.

```python
import json
import logging
import base64
from typing import Dict, Any, Optional
import boto3
from botocore.exceptions import ClientError

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("bedrock_fm_manager")

class BedrockFMManager:
    """Manages AI model invocations utilizing both Converse and InvokeModel APIs."""

    def __init__(self, region_name: str = "us-east-1"):
        # bedrock-runtime is used for all data-plane inference calls
        self.bedrock_runtime = boto3.client("bedrock-runtime", region_name=region_name)

    def converse_with_claude(self, prompt: str, model_id: str = "anthropic.claude-4.5-sonnet-v1:0") -> Optional[str]:
        """
        Uses the unified Converse API to interact with message-based models.
        Automatically maps standardized parameters.
        """
        messages = [{"role": "user", "content": [{"text": prompt}]}]
        
        # Standardized Converse inference configuration
        inference_config = {
            "maxTokens": 1024,
            "temperature": 0.5,
            "topP": 0.9
        }

        try:
            logger.info(f"Invoking {model_id} via Converse API...")
            response = self.bedrock_runtime.converse(
                modelId=model_id,
                messages=messages,
                inferenceConfig=inference_config
                # Note: Claude 4 extended 'thinking' would be passed in additionalModelRequestFields
            )
            
            output_text = response["output"]["message"]["content"][0]["text"]
            logger.info(f"Converse Success. Stop Reason: {response.get('stopReason')}")
            return output_text

        except ClientError as e:
            logger.error(f"Converse API Error: {e.response['Error']['Message']}")
            return None

    def generate_image_sdxl(self, prompt: str, output_file: str = "output.png") -> bool:
        """
        Uses the legacy InvokeModel API for SDXL, requiring a bespoke JSON payload.
        """
        model_id = "stability.stable-diffusion-xl-v1"
        
        # SDXL-specific payload structure
        payload = {
            "text_prompts": [{"text": prompt, "weight": 1.0}],
            "cfg_scale": 7.0,     # Prompt adherence strength
            "steps": 50,          # Generation steps/iterations
            "seed": 42            # Deterministic noise seed
        }

        try:
            logger.info(f"Invoking {model_id} via InvokeModel API...")
            response = self.bedrock_runtime.invoke_model(
                modelId=model_id,
                body=json.dumps(payload),
                accept="application/json",
                contentType="application/json"
            )
            
            response_body = json.loads(response["body"].read())
            
            # Extract and decode base64 image artifact
            base64_image = response_body["artifacts"][0]["base64"]
            with open(output_file, "wb") as f:
                f.write(base64.b64decode(base64_image))
                
            logger.info(f"Image successfully generated and saved to {output_file}")
            return True

        except ClientError as e:
            logger.error(f"InvokeModel API Error: {e.response['Error']['Message']}")
            return False

if __name__ == "__main__":
    manager = BedrockFMManager()
    
    # 1. Text Generation with Converse API
    result = manager.converse_with_claude("Explain the concept of an embedding vector.")
    print(f"\nClaude Output:\n{result}\n")
    
    # 2. Image Generation with InvokeModel API
    manager.generate_image_sdxl("A stylized 3D render of an AI neural network core, glowing blue.")

```